In [46]:
from __future__ import annotations
import numpy as np
import torch
from torch import nn

PATH = ""
# PATH = "storage"

Data Setup

In [47]:
import os
import sys
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split

github_path = os.path.join(PATH, "Models", "GitHub", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer
from huggingface_hub import login, hf_hub_download
from reetoolbox.loadmodels import *
# — login once per session —
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)


from reetoolbox.dataloaders import *
from reetoolbox.loadmodels import *
from reetoolbox.eval_funcs import *

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [48]:
DATASET_CLASSES = {
    "NCT":            NCTDataSet,
    "PanNuke":        PanNukeDataset,
    "PANDA":          PandaDataset,
    "PatchCamelyon":  PatchCamelyonDataset,    # ← added here
}

DATASET_PARAMS = {
    "NCT": {
        "root_dir":            os.path.join(PATH, 'data', 'NCT', 'NCT-CRC-HE-100K'),
        "batch_size":          16,
        "trainval_multiplier": 0.05,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(classification_mode='tum_vs_all'),
    },
    "PanNuke": {
        "root_dir":            os.path.join(PATH, 'data', 'PanNuke'),
        "batch_size":          16,
        "trainval_multiplier": 0.999,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(folds=(1, 2), min_positive=5),
    },
    "PANDA": {
        "root_dir":            os.path.join(PATH, 'data', "PANDA"),
        "batch_size":          16,
        "trainval_multiplier": 0.077,
        # "trainval_multiplier": 0.154,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(split='train'),
    },
    "PatchCamelyon": {
        "root_dir": os.path.join(PATH, 'data', 'PatchCamelyon'),
        "batch_size": 16,
        "trainval_multiplier": 0.0134,  # e.g. use 5% of train
        # "trainval_multiplier": 0.0268,  # e.g. use 5% of train
        "trainval_size":       0.046,  # e.g. use 5% of valid
        "extra_kwargs": {},           # no extra params needed
    },
}

In [49]:
# dataset_name = "NCT"
# dataset_name = "PanNuke"
dataset_name = "PANDA"
# dataset_name = "PatchCamelyon"

params = DATASET_PARAMS[dataset_name]
dataset_class = DATASET_CLASSES[dataset_name]

In [50]:
# NCT Parameters
mode = 'tum_vs_all'

if dataset_name == "NCT":
    dataloaders_dict = create_trainval_dict(
        dataset_class=NCTDataSet,
        root_dir=DATASET_PARAMS["NCT"]["root_dir"],
        batch_size=DATASET_PARAMS["NCT"]["batch_size"],
        transform=transforms.Compose([transforms.ToTensor()]),
        trainval_multiplier=DATASET_PARAMS["NCT"]["trainval_multiplier"],
        trainval_size=DATASET_PARAMS["NCT"]["trainval_size"],
        classification_mode=mode
    )

In [51]:
# PanNuke Parameters

if dataset_name == "PanNuke":
    dataloaders_dict = create_trainval_dict(
        dataset_class=PanNukeDataset,
        root_dir=DATASET_PARAMS["PanNuke"]["root_dir"],
        batch_size=DATASET_PARAMS["PanNuke"]["batch_size"],
        transform=transforms.Compose([transforms.ToTensor()]),
        trainval_multiplier=DATASET_PARAMS["PanNuke"]["trainval_multiplier"],
        trainval_size=DATASET_PARAMS["PanNuke"]["trainval_size"],
        folds=(1,2),
        min_positive=5
    )

In [52]:
# PANDA Parameters

if dataset_name == "PANDA":
    dataloaders_dict = create_trainval_dict(
        dataset_class=PandaDataset,
        root_dir=DATASET_PARAMS["PANDA"]["root_dir"],
        batch_size=DATASET_PARAMS["PANDA"]["batch_size"],
        transform=transforms.Compose([transforms.ToTensor()]),
        trainval_multiplier=DATASET_PARAMS["PANDA"]["trainval_multiplier"],  
        trainval_size=DATASET_PARAMS["PANDA"]["trainval_size"],        
        split='train'             
    )

Train batches: 220
 Val batches: 94


In [53]:
# PatchCamelyon train/val
if dataset_name == "PatchCamelyon":
    dataloaders_dict = create_trainval_dict(
        dataset_class=PatchCamelyonDataset,
        root_dir=DATASET_PARAMS["PatchCamelyon"]["root_dir"],
        batch_size=DATASET_PARAMS["PatchCamelyon"]["batch_size"],
        transform=transforms.Compose([transforms.ToTensor()]),
        trainval_multiplier=DATASET_PARAMS["PatchCamelyon"]["trainval_multiplier"],
        trainval_size=DATASET_PARAMS["PatchCamelyon"]["trainval_size"],
    )

In [54]:
import torch

def count_class_distribution(dataloader):
    counts = None
    for _, labels, _ in dataloader:
        # Ensure labels are 1D torch.Tensor
        bincount = torch.bincount(labels, minlength=labels.max().item() + 1)
        if counts is None:
            counts = bincount
        else:
            # If batch has more classes than previous batches, expand counts
            if bincount.numel() > counts.numel():
                counts = torch.cat([counts, torch.zeros(bincount.numel() - counts.numel(), dtype=counts.dtype)])
            counts[:bincount.numel()] += bincount
    return counts

train_counts = count_class_distribution(dataloaders_dict["train"])
val_counts = count_class_distribution(dataloaders_dict["val"])

print(f"Training: {train_counts}")
print(f"Validation: {val_counts}")

Training: tensor([1848, 1657])
Validation: tensor([793, 710])


In [55]:
import copy
import time
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

def train_loop(
    model,
    dataloaders,
    criterion,
    optimizer,
    epochs,
    device="cuda:0",
    transform_func=None,
    patience=2,   # early stopping patience in epochs
    **kwargs
):
    """
    Unified train/val loop with:
      - support for (inputs, labels, name) batches
      - early stopping based on best val ROC AUC
    """

    acc_history = []
    best_model_wts = copy.deepcopy(model.state_dict())
    best_rocauc = 0.0
    epochs_no_improve = 0
    t_full = time.time()

    for epoch in range(epochs):
        print(f'\nEpoch {epoch + 1}/{epochs}\n' + '-' * 30)
        epoch_start = time.time()

        for stage, loader in dataloaders.items():
            model.train() if stage == "train" else model.eval()

            running_loss, running_corrects, num_examples = 0.0, 0, 0
            all_labels, all_preds = [], []

            for inputs, labels, _ in tqdm(loader, desc=f"{stage}", leave=False):
                if transform_func:
                    inputs = transform_func(inputs)
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.set_grad_enabled(stage == "train"):
                    optimizer.zero_grad()

                    if transform_func:
                        inputs = transform_func(inputs)

                    outputs = model(inputs)
                    # print(outputs.requires_grad)
                    probs = torch.softmax(outputs, dim=1)[:, 1]  # binary probs for ROC
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if stage == "train":
                        loss.backward()
                        optimizer.step()

                bs = labels.size(0)
                running_loss += loss.item() * bs
                running_corrects += torch.sum(preds == labels).item()
                num_examples += bs
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(probs.detach().cpu().numpy())

            epoch_loss = running_loss / num_examples
            epoch_acc  = running_corrects / num_examples
            try:
                rocauc = roc_auc_score(all_labels, all_preds)
            except ValueError:
                rocauc = 0.0  # happens if only one class present in val set

            print(f'{stage} — Loss: {epoch_loss:.3f}  Acc: {epoch_acc:.3f}  ROC AUC: {rocauc:.3f}')

            # Track best ROC AUC on validation set
            if stage == "val":
                if rocauc > best_rocauc:
                    best_rocauc = rocauc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        acc_history.append(best_rocauc)
        print(f"Epoch time: {round(time.time() - epoch_start, 2)}s")

        # Early stopping condition
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {patience} epochs with no ROC AUC improvement.\n")
            break

    print(f"\nTraining complete. Best val ROC AUC: {best_rocauc:.3f}")
    model.load_state_dict(best_model_wts)
    return model, np.array(acc_history)

## Training

In [56]:
num_epochs = 10
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#### All ResNet Models (18, 34, 50, 101, 152)

In [57]:
resnet_datadict = create_trainval_dict(
    dataset_class=dataset_class,
    root_dir=params["root_dir"],
    batch_size=params["batch_size"],
    transform=model_transforms["ResNet18"],
    trainval_multiplier=params["trainval_multiplier"],
    trainval_size=params["trainval_size"],
    **params["extra_kwargs"]
)

Train batches: 220
 Val batches: 94


In [58]:
print("Loading ResNet18 model for training...")
resnet_NCT, optimizer_ft = load_resnet18(n_classes=2)
resnet_NCT, _ = train_loop(
    resnet_NCT, 
    resnet_datadict, 
    criterion, 
    optimizer_ft, 
    epochs=num_epochs, 
    device=device, 
    transform_func=None)

resnet_save = os.path.join(PATH, "Models", dataset_name, f"resnet18_{dataset_name}.pth")
os.makedirs(os.path.dirname(resnet_save), exist_ok=True)
torch.save(resnet_NCT, resnet_save)

c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading ResNet18 model for training...

Epoch 1/10
------------------------------


train — Loss: 0.350  Acc: 0.839  ROC AUC: 0.923


val — Loss: 0.245  Acc: 0.904  ROC AUC: 0.961
Epoch time: 17.39s

Epoch 2/10
------------------------------


train — Loss: 0.214  Acc: 0.916  ROC AUC: 0.972


val — Loss: 0.291  Acc: 0.880  ROC AUC: 0.964
Epoch time: 17.11s

Epoch 3/10
------------------------------


train — Loss: 0.089  Acc: 0.963  ROC AUC: 0.996


val — Loss: 0.304  Acc: 0.904  ROC AUC: 0.959
Epoch time: 17.54s

Epoch 4/10
------------------------------


train — Loss: 0.052  Acc: 0.982  ROC AUC: 0.999


val — Loss: 0.325  Acc: 0.906  ROC AUC: 0.963
Epoch time: 17.31s

Early stopping triggered after 2 epochs with no ROC AUC improvement.


Training complete. Best val ROC AUC: 0.964


In [59]:
# print("Loading ResNet34 model for training...")
# resnet_NCT, optimizer_ft = load_resnet34(n_classes=2)
# resnet_NCT, _ = train_loop(
#     resnet_NCT, 
#     resnet_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# resnet_save = os.path.join(PATH, "Models", dataset_name, f"resnet34_{dataset_name}.pth")
# os.makedirs(os.path.dirname(resnet_save), exist_ok=True)
# torch.save(resnet_NCT, resnet_save)

In [60]:
print("Loading ResNet50 model for training...")
resnet_NCT, optimizer_ft = load_resnet50(n_classes=2)
resnet_NCT, _ = train_loop(
    resnet_NCT, 
    resnet_datadict, 
    criterion, 
    optimizer_ft, 
    epochs=num_epochs, 
    device=device, 
    transform_func=None)

resnet_save = os.path.join(PATH, "Models", dataset_name, f"resnet50_{dataset_name}.pth")
os.makedirs(os.path.dirname(resnet_save), exist_ok=True)
torch.save(resnet_NCT, resnet_save)

c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading ResNet50 model for training...

Epoch 1/10
------------------------------


train — Loss: 0.339  Acc: 0.855  ROC AUC: 0.928


val — Loss: 0.228  Acc: 0.908  ROC AUC: 0.967
Epoch time: 27.66s

Epoch 2/10
------------------------------


train — Loss: 0.190  Acc: 0.922  ROC AUC: 0.978


...

val — Loss: 0.262  Acc: 0.911  ROC AUC: 0.968
Epoch time: 26.11s

Epoch 4/10
------------------------------


train — Loss: 0.064  Acc: 0.979  ROC AUC: 0.997


val — Loss: 0.333  Acc: 0.911  ROC AUC: 0.965
Epoch time: 25.86s

Epoch 5/10
------------------------------


train — Loss: 0.050  Acc: 0.985  ROC AUC: 0.998


val — Loss: 0.338  Acc: 0.912  ROC AUC: 0.966
Epoch time: 25.53s

Early stopping triggered after 2 epochs with no ROC AUC improvement.


Training complete. Best val ROC AUC: 0.968


In [ ]:
# print("Loading ResNet101 model for training...")
# resnet_NCT, optimizer_ft = load_resnet101(n_classes=2)
# resnet_NCT, _ = train_loop(
#     resnet_NCT, 
#     resnet_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# resnet_save = os.path.join(PATH, "Models", dataset_name, f"resnet101_{dataset_name}.pth")
# os.makedirs(os.path.dirname(resnet_save), exist_ok=True)
# torch.save(resnet_NCT, resnet_save)

In [ ]:
# print("Loading ResNet152 model for training...")
# resnet_NCT, optimizer_ft = load_resnet152(n_classes=2)
# resnet_NCT, _ = train_loop(
#     resnet_NCT, 
#     resnet_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# resnet_save = os.path.join(PATH, "Models", dataset_name, f"resnet152_{dataset_name}.pth")
# os.makedirs(os.path.dirname(resnet_save), exist_ok=True)
# torch.save(resnet_NCT, resnet_save)

#### UNI and UNI2

In [ ]:
# UNI_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["UNI"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# print("Loading UNI model for training...")
# uni_NCT, optimizer_ft = load_uni(n_classes=2)
# uni_NCT, _ = train_loop(
#     uni_NCT, 
#     UNI_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# uni_save = os.path.join(PATH, "Models", dataset_name, f"uni_{dataset_name}.pth")
# os.makedirs(os.path.dirname(uni_save), exist_ok=True)
# torch.save(uni_NCT.head.state_dict(), uni_save)

In [ ]:
# print("Loading UNI2 model for training...")
# uni2_NCT, optimizer_ft = load_uni2(n_classes=2)
# uni2_NCT, _ = train_loop(
#     uni2_NCT, 
#     UNI_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# uni2_save = os.path.join(PATH, "Models", dataset_name, f"uni2_{dataset_name}.pth")
# os.makedirs(os.path.dirname(uni2_save), exist_ok=True)
# torch.save(uni2_NCT.head.state_dict(), uni2_save)

#### GigaPath

In [ ]:
# GigaPath_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["GigaPath"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# print("Loading GigaPath model for training...")
# gigapath_NCT, optimizer_ft = load_gigapath(n_classes=2)
# gigapath_NCT, _ = train_loop(
#     gigapath_NCT, 
#     GigaPath_datadict, 
#     criterion, 
#     optimizer_ft, 
#     epochs=num_epochs, 
#     device=device, 
#     transform_func=None)

# gigapath_save = os.path.join(PATH, "Models", dataset_name, f"gigapath_{dataset_name}.pth")
# os.makedirs(os.path.dirname(gigapath_save), exist_ok=True)
# torch.save(gigapath_NCT.head.state_dict(), gigapath_save)

#### Virchow

In [ ]:
# # ─── Virchow training run ────────────────────────────────────────────
# from timm.data import resolve_data_config, create_transform

# print("Loading Virchow model for training...")
# vir_NCT, optimizer_ft = load_virchow(n_classes=2, device=device)

# data_cfg = resolve_data_config(vir_NCT.backbone.pretrained_cfg, model=vir_NCT.backbone)
# vir_transform = create_transform(**data_cfg)

# vir_NCT, _ = train_loop(
#     vir_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=vir_transform
# )

# vir_save = os.path.join(PATH, "Models", dataset_name, f"virchow_{dataset_name}.pth")
# os.makedirs(os.path.dirname(vir_save), exist_ok=True)
# torch.save(vir_NCT.head.state_dict(), vir_save)

#### Virchow2

In [ ]:
# # ─── Virchow-2 training run ─────────────────────────────────────────
# from timm.data import resolve_data_config, create_transform

# print("Loading Virchow-2 model for training...")
# vir_NCT, optimizer_ft = load_virchow2(n_classes=2, device=device)

# # Build the official timm transform once
# data_cfg = resolve_data_config(vir_NCT.backbone.pretrained_cfg,
#                                model=vir_NCT.backbone)
# vir_transform = create_transform(**data_cfg)

# vir_NCT, _ = train_loop(
#     vir_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=vir_transform      # <-- apply identical preprocessing
# )

# vir_save = os.path.join(PATH, "Models", dataset_name, f"virchow2_{dataset_name}.pth")
# os.makedirs(os.path.dirname(vir_save), exist_ok=True)
# torch.save(vir_NCT.head.state_dict(), vir_save)

#### H0-mini

In [ ]:
# h0mini_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["H0-mini"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# print("Loading H0-mini model for training...")
# h0mini_NCT, optimizer_ft = load_h0_mini(n_classes=2, device=device)

# h0mini_NCT, _ = train_loop(
#     h0mini_NCT,
#     h0mini_datadict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=None
# )

# hopt_save = os.path.join(PATH, "Models", dataset_name, f"h0_mini_{dataset_name}.pth")
# os.makedirs(os.path.dirname(hopt_save), exist_ok=True)
# torch.save(h0mini_NCT.head.state_dict(), hopt_save)

#### H-Optimus-0

In [ ]:
# Hop0_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["H-Optimus-0"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# # ─── H-Optimus-0 ────────────────────────────────────────────────────
# print("Loading H-Optimus model for training...")
# hoptimus_NCT, optimizer_ft = load_h_optimus0(n_classes=2, device=device)

# hoptimus_NCT, _ = train_loop(
#     hoptimus_NCT,
#     Hop0_datadict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=None
# )

# hopt_save = os.path.join(PATH, "Models", dataset_name, f"hoptimus0_{dataset_name}.pth")
# os.makedirs(os.path.dirname(hopt_save), exist_ok=True)
# torch.save(hoptimus_NCT.head.state_dict(), hopt_save)

#### H-Optimus-1

In [ ]:
# Hop1_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["H-Optimus-1"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# # ─── H-Optimus-1 ────────────────────────────────────────────────────
# print("Loading H-Optimus model for training...")
# hoptimus_NCT, optimizer_ft = load_h_optimus1(n_classes=2, device=device)

# hoptimus_NCT, _ = train_loop(
#     hoptimus_NCT,
#     Hop1_datadict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=None
# )

# hopt_save = os.path.join(PATH, "Models", dataset_name, f"hoptimus1_{dataset_name}.pth")
# os.makedirs(os.path.dirname(hopt_save), exist_ok=True)
# torch.save(hoptimus_NCT.head.state_dict(), hopt_save)

#### EXAONEPath

In [ ]:
# EXAONE_datadict = create_trainval_dict(
#     dataset_class=dataset_class,
#     root_dir=params["root_dir"],
#     batch_size=params["batch_size"],
#     transform=model_transforms["EXAONEPath"],
#     trainval_multiplier=params["trainval_multiplier"],
#     trainval_size=params["trainval_size"],
#     **params["extra_kwargs"]
# )

In [ ]:
# # ─── EXAONEPath ────────────────────────────────────────────────────
# print("Loading EXAONEPath model for training...")
# exaone_NCT, optimizer_ft = load_exaonepath(n_classes=2, device=device)

# exaone_NCT, _ = train_loop(
#     exaone_NCT,
#     EXAONE_datadict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=None
# )

# exaone_save = os.path.join(PATH, "Models", dataset_name, f"exaonepath_{dataset_name}.pth")
# os.makedirs(os.path.dirname(exaone_save), exist_ok=True)
# torch.save(exaone_NCT.head.state_dict(), exaone_save)

#### Hibou-B and Hibou-L

In [ ]:
# # ─── Hibou-B ───────────────────────────────────────────────────────
# print("Loading Hibou-B model for training...")
# processor_hibou_b, hibou_b_NCT, optimizer_ft = load_hibou_b(n_classes=2, device=device)

# # wrap the HF processor as a simple transform (img → tensor)
# def hibou_b_transform(img):
#     if isinstance(img, torch.Tensor):
#         # Assume it's already been transformed
#         return img
#     else:
#         pixel_values = processor_hibou_b(images=img, return_tensors="pt").pixel_values.squeeze(0)
#         return pixel_values

# hibou_b_NCT, _ = train_loop(
#     hibou_b_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=hibou_b_transform
# )

# hibou_b_save = os.path.join(PATH, "Models", dataset_name, f"hibou_b_{dataset_name}.pth")
# os.makedirs(os.path.dirname(hibou_b_save), exist_ok=True)
# torch.save(hibou_b_NCT.head.state_dict(), hibou_b_save)

In [ ]:
# # ─── Hibou-B ───────────────────────────────────────────────────────
# print("Loading Hibou-B model for training...")
# processor_hibou_l, hibou_l_NCT, optimizer_ft = load_hibou_l(n_classes=2, device=device)

# # wrap the HF processor as a simple transform (img → tensor)
# def hibou_l_transform(img):
#     if isinstance(img, torch.Tensor):
#         # Assume it's already been transformed
#         return img
#     else:
#         pixel_values = processor_hibou_l(images=img, return_tensors="pt").pixel_values.squeeze(0)
#         return pixel_values

# hibou_l_NCT, _ = train_loop(
#     hibou_l_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=hibou_l_transform
# )

# hibou_l_save = os.path.join(PATH, "Models", dataset_name, f"hibou_l_{dataset_name}.pth")
# os.makedirs(os.path.dirname(hibou_l_save), exist_ok=True)
# torch.save(hibou_l_NCT.head.state_dict(), hibou_l_save)

#### Phikon v1

In [ ]:
# # ─── Phikon (v1) ───────────────────────────────────────────────────
# print("Loading Phikon v1 model for training...")
# processor_phikon, phikon_NCT, optimizer_ft = load_phikon(n_classes=2, device=device)

# def phikon_transform(img):
#     img = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)(img)
#     if isinstance(img, torch.Tensor):
#         return img  # Already transformed
#     else:
#         pixel_values = processor_phikon(images=img, return_tensors="pt").pixel_values.squeeze(0)
#         return pixel_values

# phikon_NCT, _ = train_loop(
#     phikon_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=phikon_transform
# )

# phikon_save = os.path.join(PATH, "Models", dataset_name, f"phikon_{dataset_name}.pth")
# os.makedirs(os.path.dirname(phikon_save), exist_ok=True)
# torch.save(phikon_NCT.head.state_dict(), phikon_save)

#### Phikon v2

In [ ]:
# # ─── Phikon-v2 ─────────────────────────────────────────────────────
# print("Loading Phikon v2 model for training...")
# processor_pk2, pk2_NCT, optimizer_ft = load_phikon_v2(n_classes=2, device=device)

# def pk2_transform(img):
#     img = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)(img)
#     if isinstance(img, torch.Tensor):
#         return img
#     else:
#         pixel_values = processor_pk2(images=img, return_tensors="pt").pixel_values.squeeze(0)
#         return pixel_values

# pk2_NCT, _ = train_loop(
#     pk2_NCT,
#     dataloaders_dict,
#     criterion,
#     optimizer_ft,
#     epochs=num_epochs,
#     device=device,
#     transform_func=pk2_transform
# )

# pk2_save = os.path.join(PATH, "Models", dataset_name, f"phikon_v2_{dataset_name}.pth")
# os.makedirs(os.path.dirname(pk2_save), exist_ok=True)
# torch.save(pk2_NCT.head.state_dict(), pk2_save)

In [ ]:
# # Assumes your load_* functions are already imported
# import torch

# device = "cuda" if torch.cuda.is_available() else "cpu"
# n_classes = 2

# models_to_check = [
#     ("H-Optimus-0", load_h_optimus),
#     ("Hibou-B", lambda n_classes, device: (load_hibou_b(n_classes, device)[1], load_hibou_b(n_classes, device)[2])),
#     ("EXAONEPath", load_exaonepath),
#     ("Phikon-v1", lambda n_classes, device: (load_phikon(n_classes, device)[1], load_phikon(n_classes, device)[2])),
#     ("Phikon-v2", lambda n_classes, device: (load_phikon_v2(n_classes, device)[1], load_phikon_v2(n_classes, device)[2])),
#     ("Virchow-2", load_virchow2),
# ]

# for name, loader in models_to_check:
#     print(f"\n{name}:")
#     model, _ = loader(n_classes, device)
#     total_params = sum(p.numel() for p in model.parameters())
#     trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     print(f"  Total parameters: {total_params:,}")
#     print(f"  Trainable parameters: {trainable_params:,}")